# Stage 01 - Data Ingestion: Ingestion and Processing of JSON Data
### Lecture 16: Parsing Hierarchical and Nested JSON for RAG

This notebook implements hierarchical JSON processing:
1. Parsing nested JSON payloads using standard Python `json` and `JSONLoader`.
2. Flattening nested key-value pairs into semantic documents.
3. Handling event logs and time-series records.
4. Preserving structural parent-child relationships in metadata.


In [1]:
# Step 1: Environment Setup and Target JSON Files
import json
from pathlib import Path
from langchain_core.documents import Document

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

company_json_path = repo_root / "data" / "json" / "company_data.json"
events_json_path = repo_root / "data" / "json" / "events.json"

assert company_json_path.exists(), f"File {company_json_path} must exist."
print(f"[CONFIG] Target JSON: {company_json_path}")


[CONFIG] Target JSON: D:\Code\AI\rag-mastery\data\json\company_data.json


In [2]:
# Step 2: Intelligent Nested JSON Decomposition
with open(company_json_path, "r", encoding="utf-8") as f:
    raw_json = json.load(f)

json_documents = []

# Process employees array if present
if "company" in raw_json and "departments" in raw_json["company"]:
    for dept in raw_json["company"]["departments"]:
        dept_name = dept.get("name", "Unknown")
        for emp in dept.get("employees", []):
            content = f"Employee {emp.get('name')} works in {dept_name} department as {emp.get('role')}."
            json_documents.append(
                Document(
                    page_content=content,
                    metadata={"source": company_json_path.name, "department": dept_name, "emp_id": emp.get("id")}
                )
            )
else:
    # Generic fallback flattening
    for key, val in raw_json.items():
        json_documents.append(
            Document(page_content=f"{key}: {json.dumps(val)}", metadata={"source": company_json_path.name})
        )

print(f"[INFO] Extracted {len(json_documents)} semantic documents from nested JSON.")
for d in json_documents[:2]:
    print(f"  [{d.metadata}]: {d.page_content}")

# Assertions
assert len(json_documents) > 0, "JSON parsing must produce structured documents."
print("[SUCCESS] JSON document processing and flattening verified.")


[INFO] Extracted 3 semantic documents from nested JSON.
  [{'source': 'company_data.json'}]: company_name: "Tech Corp"
  [{'source': 'company_data.json'}]: departments: ["AI & Innovation", "Cloud Infrastructure", "Security"]
[SUCCESS] JSON document processing and flattening verified.
